# Persona drift × Jacobian lens: one-click Colab run

Runs `experiments/persona_drift/run.py` end to end on **Qwen2.5-7B-Instruct**: fit lens → extract assistant axis → steering sweep → J-space readout → figures.

1. **Runtime → Change runtime type → L4 GPU** (24 GB). An A100 also works. The free T4 (16 GB) is too small for 7B.
2. *(Optional, recommended)* Open **🔑 Secrets** in the left sidebar, add `OPENAI_API_KEY`, and turn on notebook access. The assistant-axis judge (gpt-4.1-mini) then keeps only role-playing samples. Without it, every role sample is kept and the axis is noisier.
3. **Runtime → Run all.** Approve the Google Drive prompt. Outputs go to Drive.

**Disconnected?** Reconnect and choose *Run all* again. Each stage skips work it has already done, and the lens fit checkpoints after each prompt.

Expected time on an L4 with the defaults below: about 3–5 h, mostly the lens fit and axis generation (not yet measured on this GPU).

In [ ]:
# @title Settings
MODEL = "Qwen/Qwen2.5-7B-Instruct"  # @param {type:"string"}
USE_DRIVE = True  # @param {type:"boolean"}
OUT_DIR = "/content/drive/MyDrive/persona_drift/qwen2.5-7b"  # @param {type:"string"}
N_FIT_PROMPTS = 100  # @param {type:"integer"}
# @markdown Assistant-axis extraction: `AXIS_N_ROLES` roles sampled with a fixed seed from the 275 in `role_list.json` (0 = all). Each role gets 5 instruction variants × `AXIS_QUESTION_COUNT` questions.
AXIS_N_ROLES = 100  # @param {type:"integer"}
AXIS_QUESTION_COUNT = 10  # @param {type:"integer"}
AXIS_MAX_TOKENS = 256  # @param {type:"integer"}
AXIS_BATCH_SIZE = 32  # @param {type:"integer"}
MIN_COUNT = 10  # @param {type:"integer"}
STRENGTHS = "-2 -1 -0.5 0 0.5 1 2"  # @param {type:"string"}
BRANCH = "devin/1795312846-persona-drift"  # @param {type:"string"}

In [ ]:
# @title Check GPU
import subprocess

import torch

assert torch.cuda.is_available(), "No GPU: Runtime → Change runtime type → L4 GPU"
name = torch.cuda.get_device_name(0)
mem = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f"{name}: {mem:.1f} GB")
assert mem > 20, f"{name} has {mem:.0f} GB; a 7B run needs ≥24 GB (pick L4 or A100)"

In [ ]:
# @title Mount Drive, fetch code, install
import os

if USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
else:
    OUT_DIR = "/content/out"
os.makedirs(OUT_DIR, exist_ok=True)


def sh(cmd):
    print("$", cmd)
    subprocess.run(cmd, shell=True, check=True)


JL, AA = "/content/jacobian-lens", "/content/assistant-axis"
if not os.path.exists(JL):
    r = subprocess.run(
        f"git clone -q --depth 1 -b {BRANCH} https://github.com/izzimaria/jacobian-lens {JL}",
        shell=True,
    )
    if r.returncode != 0:  # branch merged/deleted
        sh(f"git clone -q --depth 1 https://github.com/izzimaria/jacobian-lens {JL}")
if not os.path.exists(AA):
    sh(f"git clone -q --depth 1 https://github.com/izzimaria/assistant-axis {AA}")
sh(
    'pip install -q "transformers>=5.5" accelerate datasets jsonlines openai python-dotenv scikit-learn plotly pyarrow'
)
sh(f"pip install -q -e {JL}")
sh(f"pip install -q --no-deps -e {AA}")

In [ ]:
# @title OpenAI judge key (optional, from Colab Secrets)
JUDGE = "none"
try:
    from google.colab import userdata

    key = userdata.get("OPENAI_API_KEY")
    if key:
        os.environ["OPENAI_API_KEY"] = key
        JUDGE = "openai"
except Exception as e:
    print("no OPENAI_API_KEY secret:", type(e).__name__)
print("judge:", JUDGE)

In [ ]:
# @title Role subset (saved to Drive so a resumed run uses the same roles)
import json
import random

roles_file = f"{OUT_DIR}/axis_roles.json"
if os.path.exists(roles_file):
    ROLES = json.load(open(roles_file))
else:
    all_roles = sorted(json.load(open(f"{AA}/data/roles/role_list.json")))
    ROLES = (
        all_roles
        if AXIS_N_ROLES <= 0
        else sorted(random.Random(0).sample(all_roles, AXIS_N_ROLES))
    )
    json.dump(ROLES, open(roles_file, "w"))
print(len(ROLES), "roles:", ", ".join(ROLES[:12]), "...")

In [ ]:
# @title Run the pipeline (fit → axis → sweep → readout → analyze)
import shlex
import sys

cmd = [
    sys.executable,
    "-u",
    f"{JL}/experiments/persona_drift/run.py",
    "--model",
    MODEL,
    "--assistant_axis_dir",
    AA,
    "--out",
    OUT_DIR,
    "--n_fit_prompts",
    str(N_FIT_PROMPTS),
    "--dim_batch",
    "8",
    "--axis_roles",
    *ROLES,
    "--axis_question_count",
    str(AXIS_QUESTION_COUNT),
    "--axis_max_tokens",
    str(AXIS_MAX_TOKENS),
    "--axis_batch_size",
    str(AXIS_BATCH_SIZE),
    "--min_count",
    str(MIN_COUNT),
    "--judge",
    JUDGE,
    "--generator",
    "hf",
    "--strengths",
    *STRENGTHS.split(),
]
print(shlex.join(cmd[:12]), "...")
with open(f"{OUT_DIR}/run.log", "a") as log:
    p = subprocess.Popen(
        cmd, cwd=JL, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    for line in p.stdout:
        if "HTTP Request" in line:
            continue
        print(line, end="")
        log.write(line)
    assert p.wait() == 0, f"run.py failed (exit {p.returncode}); see {OUT_DIR}/run.log"
print("done →", OUT_DIR)

In [ ]:
# @title Results
import pandas as pd
from IPython.display import HTML, Image, display

summary = json.load(open(f"{OUT_DIR}/summary.json"))
cols = [
    "proj_steer_layer",
    "persona_band_hit_rate",
    "persona_band_median_rank",
    "persona_emitted_rate",
    "assistant_band_median_rank",
    "persona_frac_band_before_emit",
    "persona_median_lead_tokens",
]
df = pd.DataFrame(summary["headline"]).T
display(df[[c for c in cols if c in df]].round(3))
print("onset layers:", json.dumps(summary.get("onset_layers"), indent=1))
for f in sorted(os.listdir(f"{OUT_DIR}/figures")):
    print(f)
    display(Image(f"{OUT_DIR}/figures/{f}"))

In [ ]:
# @title Slice page (most negative s)
import html

slices = sorted(os.listdir(f"{OUT_DIR}/slices"))
print(slices)
page = open(f"{OUT_DIR}/slices/{slices[0]}").read()
display(
    HTML(f'<iframe srcdoc="{html.escape(page)}" width="100%" height="700"></iframe>')
)

Full figure notebook: open `experiments/persona_drift/persona_drift.ipynb` with `PERSONA_DRIFT_OUT` set to `OUT_DIR`. To pick up the latest code, delete `/content/jacobian-lens` and re-run the install cell. `lens.pt` and `axis.pt` stay on Drive and are reused.